The CDC data summarizes the health measures of the census tract, and the LocationName field contains the FIPS code of each tract which can be used to join on MSSA data fields.

In [0]:
cdc_places_bronze_df = spark.read.table("ca_healthcare_fac_bronze.cdc_places_ca_data.cdc_places_ca_data").toPandas()
print(cdc_places_bronze_df.columns)

In [0]:
mssa_bronze_pddf = spark.read.table("ca_healthcare_fac_bronze.mssa_data_bronze.mssa_geo").toPandas()
print(mssa_bronze_pddf.columns)

Merge on MSSA data but the geometry rings can't be joined in this stage due to server memory shortage

In [0]:
cdc_places_w_mssa = cdc_places_bronze_df.merge(mssa_bronze_pddf[['TRACTCE', 'GEOID', 'MSSAID', 'MSSANM']], left_on='LocationName', right_on='GEOID', how='left')

Data value Unit column only has % signs. Whic hindicates the variable Data Value represents percentages. Similarly, Data Value Footnote and Data Value Footnote Symbol columns don't have any values. Need to drop them along with the GEOID field coming from the MSSA data which is redundant at this point

In [0]:
cdc_places_final = cdc_places_w_mssa.drop(columns=['Data_Value_Unit', 'Data_Value_Footnote', 'Data_Value_Footnote_Symbol', "GEOID"])
cdc_places_final.head()


In [0]:
cdc_places_final.shape

In [0]:
cdc_places_spark = spark.createDataFrame(cdc_places_final)

In [0]:
cdc_places_spark.write.mode('overwrite').saveAsTable('ca_healthcare_fac_silver.cdc_places_silver.cdc_places_mssa_v0')

To Download the file for handoff, save .csv file in the volume

In [0]:
volume_path = "/Volumes/ca_healthcare_fac_silver/default/silver_export"

cdc_places_final.to_csv(f"{volume_path}/cdc_places_silver.csv", index=False)
